# Fixed-expiry Half-Skew strategy comparison

本Notebook对六个固定期限分别回测，同时把Call Skew和Put Skew设置为非零软目标。
Wing Curvature仅作为由两侧Half Skew派生的诊断量，不作为独立优化因子。

## 当前模块参数值

参数默认值统一在 `00_config.ipynb` 设置；本 cell 只打印当前内核中的实际值。


In [1]:
_ipython = get_ipython() if 'get_ipython' in globals() else None
if _ipython is not None:
    _ipython.run_line_magic('run', './00_config.ipynb')

_module_parameter_names = [
    'SKEW_STRATEGY_OUTPUT_PATH','VOL_MODEL','MODEL_SKEW_TAG','HALF_SKEW_H',
    'STRATEGY_INCLUDE_ATM_OPTIONS','STRATEGY_ATM_STRIKES_PER_SIDE',
    'STRATEGY_HALF_SKEW_TARGET_SWITCH_DATE','STRATEGY_HALF_SKEW_TARGETS_BEFORE',
    'STRATEGY_HALF_SKEW_TARGETS_AFTER','STRATEGY_HEDGE_GAMMA','STRATEGY_HEDGE_ATM_VOL',
    'STRATEGY_HEDGE_ATMVOL_VANNA','STRATEGY_HEDGE_THETA','STRATEGY_POSITION_BOUND',
    'STRATEGY_MAX_DAILY_TRADE_PER_LEG','STRATEGY_MIN_OPTION_VOLUME',
    'STRATEGY_USE_PREMIUM_MARGIN_FILTER','STRATEGY_MIN_PREMIUM_MARGIN_RATIO',
    'STRATEGY_MAX_ABS_LOG_MONEYNESS','STRATEGY_MAX_MODEL_IV',
    'STRATEGY_MIN_CANDIDATE_OPTIONS','STRATEGY_MIN_DTE_DAYS','STRATEGY_RIDGE',
    'STRATEGY_CAPITAL_PENALTY',
    'STRATEGY_INTEGER_OPTION_POSITIONS','STRATEGY_INTEGER_FUTURES_POSITIONS',
]
print('06_skew_strategy.ipynb 当前参数：')
for _parameter_name in _module_parameter_names:
    print(f'{_parameter_name} = {globals()[_parameter_name]!r}')

/opt/anaconda3/lib/python3.13/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


配置参数数量: 71


,参数,含义,默认值,状态
0,PROJECT_PATH,项目目录,PosixPath('/Users/mac/Desktop/实习/half skew str...,已设置
1,UNDERLYING_DATA_PATH,标的原始数据路径,PosixPath('数据/标的_2026-06~07.csv'),已设置
2,FUTURE_DATA_PATH,IM期货原始数据路径；当前与标的数据共用文件,PosixPath('数据/标的_2026-06~07.csv'),已设置
3,OPTION_DATA_PATH,MO期权原始数据路径,PosixPath('数据/MO_2026-06~07.csv'),已设置
4,OUTPUT_PATH,通用输出目录,PosixPath('/Users/mac/Desktop/实习/half skew str...,已设置
...,...,...,...,...
66,STRATEGY_OPTION_MARGIN_RATE,卖出期权保证金中的标的保证金率,0.12,已设置
67,STRATEGY_MINIMUM_MARGIN_RATE,卖出期权最低保证金率,0.06,已设置
68,STRATEGY_INTEGER_OPTION_POSITIONS,是否将06策略期权仓位取整,True,已设置
69,STRATEGY_INTEGER_FUTURES_POSITIONS,是否将06策略期货对冲仓位取整,True,已设置


06_skew_strategy.ipynb 当前参数：
SKEW_STRATEGY_OUTPUT_PATH = PosixPath('/Users/mac/Desktop/实习/half skew strategy 8.24 占资问题/outputs/06_skew_strategy_QUADRATIC_HALF-SKEW_CURVATURE-WING')
VOL_MODEL = 'QUADRATIC'
MODEL_SKEW_TAG = 'QUADRATIC_HALF-SKEW_CURVATURE-WING'
HALF_SKEW_H = 0.1
STRATEGY_INCLUDE_ATM_OPTIONS = False
STRATEGY_ATM_STRIKES_PER_SIDE = 2
STRATEGY_HALF_SKEW_TARGET_SWITCH_DATE = '2026-07-17'
STRATEGY_HALF_SKEW_TARGETS_BEFORE = {'2606': {'CALL_SKEW': -20000.0, 'PUT_SKEW': 20000.0}, '2607': {'CALL_SKEW': -20000.0, 'PUT_SKEW': 20000.0}, '2608': {'CALL_SKEW': 20000.0, 'PUT_SKEW': -20000.0}, '2609': {'CALL_SKEW': 20000.0, 'PUT_SKEW': -20000.0}, '2612': {'CALL_SKEW': 20000.0, 'PUT_SKEW': -20000.0}, '2703': {'CALL_SKEW': -20000.0, 'PUT_SKEW': 20000.0}}
STRATEGY_HALF_SKEW_TARGETS_AFTER = {'2606': {'CALL_SKEW': -20000.0, 'PUT_SKEW': 20000.0}, '2607': {'CALL_SKEW': -20000.0, 'PUT_SKEW': 20000.0}, '2608': {'CALL_SKEW': -20000.0, 'PUT_SKEW': 20000.0}, '2609': {'CALL_SKEW': -20000.0, 'PUT_SKE

## 1. 依赖、参数与数据口径

输入来自 01、03、04 的 CSV；核心输出是六个固定期限的每日 PnL、持仓、保证金、Traditional Taylor 和完整 T2 归因及诊断图。


In [2]:
from __future__ import annotations
import itertools, math, warnings
from pathlib import Path
from typing import Dict, Tuple, TypeAlias
import numpy as np
import pandas as pd
from scipy.optimize import lsq_linear
from scipy.special import ndtr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.axes import Axes
from matplotlib.ticker import PercentFormatter

DataRow: TypeAlias = pd.Series | tuple
ScalarOrArray: TypeAlias = float | np.ndarray

# 优先复用00--04在当前内核生成的对象；单独运行本Notebook时读取8.13自身CSV。
if 'option_forward_panel' in globals():
    OPT = option_forward_panel.copy()
else:
    _repo_path = Path(globals().get('REPO_FORWARD_OUTPUT_PATH', Path.cwd() / 'outputs' / '03_repo_forward'))
    OPT = pd.read_csv(_repo_path / 'option_forward_panel.csv')

# All tunable strategy parameters are defined in 00_config.ipynb.
# Explicit errors prevent a standalone Step-6 run from silently using stale defaults.
_REQUIRED_STRATEGY_PARAMETERS = [
    'HALF_SKEW_H','STRATEGY_HALF_SKEW_TARGET_SWITCH_DATE',
    'STRATEGY_HALF_SKEW_TARGETS_BEFORE','STRATEGY_HALF_SKEW_TARGETS_AFTER',
    'STRATEGY_EXPIRY_CODES','STRATEGY_MAX_OPTIONS','STRATEGY_INCLUDE_ATM_OPTIONS',
    'STRATEGY_ATM_STRIKES_PER_SIDE','STRATEGY_HEDGE_GAMMA',
    'STRATEGY_HEDGE_ATM_VOL','STRATEGY_HEDGE_THETA','STRATEGY_HEDGE_ATMVOL_VANNA',
    'STRATEGY_POSITION_BOUND','STRATEGY_MAX_DAILY_TRADE_PER_LEG',
    'STRATEGY_MIN_OPTION_VOLUME','STRATEGY_USE_PREMIUM_MARGIN_FILTER',
    'STRATEGY_MIN_PREMIUM_MARGIN_RATIO','STRATEGY_MAX_ABS_LOG_MONEYNESS',
    'STRATEGY_MAX_MODEL_IV','STRATEGY_MIN_CANDIDATE_OPTIONS','STRATEGY_MIN_DTE_DAYS',
    'STRATEGY_RIDGE','STRATEGY_CAPITAL_PENALTY','STRATEGY_OPTION_MULTIPLIER','STRATEGY_OPTION_FEE_PER_CONTRACT',
    'STRATEGY_FUTURES_MULTIPLIER','STRATEGY_FUTURES_FEE_RATE',
    'STRATEGY_FUTURES_MARGIN_RATE','STRATEGY_OPTION_MARGIN_RATE',
    'STRATEGY_MINIMUM_MARGIN_RATE','STRATEGY_TAYLOR_STEP',
    'STRATEGY_INTEGER_OPTION_POSITIONS','STRATEGY_INTEGER_FUTURES_POSITIONS',
]
_missing=[name for name in _REQUIRED_STRATEGY_PARAMETERS if name not in globals()]
if _missing:
    raise RuntimeError('Run 00_config.ipynb before Step 6; missing: '+', '.join(_missing))
if float(HALF_SKEW_H) <= 0:
    raise ValueError('HALF_SKEW_H must be positive')
if int(STRATEGY_ATM_STRIKES_PER_SIDE) < 0:
    raise ValueError('STRATEGY_ATM_STRIKES_PER_SIDE must be non-negative')
if bool(STRATEGY_INCLUDE_ATM_OPTIONS) and 2 * int(STRATEGY_ATM_STRIKES_PER_SIDE) > int(STRATEGY_MAX_OPTIONS):
    raise ValueError('2 * STRATEGY_ATM_STRIKES_PER_SIDE cannot exceed STRATEGY_MAX_OPTIONS')
for _target_name in ['STRATEGY_HALF_SKEW_TARGETS_BEFORE','STRATEGY_HALF_SKEW_TARGETS_AFTER']:
    _target_table=globals()[_target_name]
    for _expiry_code in map(str,STRATEGY_EXPIRY_CODES):
        if _expiry_code not in _target_table:
            raise KeyError(f'{_target_name} missing expiry {_expiry_code}')
        if set(_target_table[_expiry_code]) != {'CALL_SKEW','PUT_SKEW'}:
            raise KeyError(f'{_target_name}[{_expiry_code}] must contain CALL_SKEW and PUT_SKEW')

M = float(STRATEGY_OPTION_MULTIPLIER)
FEE = float(STRATEGY_OPTION_FEE_PER_CONTRACT)
FUTURES_MULTIPLIER = float(STRATEGY_FUTURES_MULTIPLIER)
FUTURES_FEE_RATE = float(STRATEGY_FUTURES_FEE_RATE)
FUTURES_MARGIN_RATE = float(STRATEGY_FUTURES_MARGIN_RATE)

# ========================== User interfaces ==========================
# Volatility model used for selection, pricing and attribution.
# Allowed values: 'QUADRATIC' and 'SVI'; defaults to the model selected in 00_config.
VOLATILITY_MODEL = str(globals().get('VOL_MODEL', 'QUADRATIC')).upper()
_model_tag = globals().get(
    'MODEL_SKEW_TAG',
    f'{VOLATILITY_MODEL}_SKEW-DERIVATIVE_CURVATURE-DERIVATIVE',
)
_project_path = Path(globals().get('PROJECT_PATH', Path.cwd())).resolve()
_output_root = Path(globals().get('OUTPUT_PATH', _project_path / 'outputs'))
VOLATILITY_PARAMETER_FILES = {
    VOLATILITY_MODEL: str(
        _output_root / f'04_volatility_model_{_model_tag}'
        / 'volatility_model_parameters.csv'
    ),
}

# Short aliases used by the implementation; values come from 00_config.ipynb.
MAX_OPTIONS = int(STRATEGY_MAX_OPTIONS)
OPTION_MARGIN_RATE = float(STRATEGY_OPTION_MARGIN_RATE)
MINIMUM_MARGIN_RATE = float(STRATEGY_MINIMUM_MARGIN_RATE)
if float(STRATEGY_MIN_PREMIUM_MARGIN_RATIO) < 0:
    raise ValueError('STRATEGY_MIN_PREMIUM_MARGIN_RATIO must be non-negative')

## 2. 输入数据与波动率曲面

读取 01、03、04 的标准化面板，并把不同模型参数统一成局部 ATM、Skew、Curvature 因子。


In [3]:
def load_volatility_parameters(model_name: str) -> pd.DataFrame:
    """读取并校验指定波动率模型参数，同时补齐利率字段。"""
    model_name = str(model_name).upper()
    if model_name not in VOLATILITY_PARAMETER_FILES:
        raise ValueError("VOLATILITY_MODEL must be 'QUADRATIC' or 'SVI'")
    path = Path(VOLATILITY_PARAMETER_FILES[model_name])
    if not path.exists():
        raise FileNotFoundError(
            f'{model_name} parameter CSV not found: {path}. '
            'Run 04_volatility_model.ipynb with the same VOL_MODEL first.'
        )
    parameters = pd.read_csv(path)
    required = ({'TRADE_DT','EXPIRY','TAU','a','b','c'} if model_name == 'QUADRATIC'
                else {'TRADE_DT','EXPIRY','TAU','a','b','rho','m','sigma'})
    missing = required-set(parameters.columns)
    if missing:
        raise ValueError(f'{model_name} parameter CSV missing columns: {sorted(missing)}')
    if 'MODEL' in parameters.columns:
        wrong = parameters.loc[~parameters['MODEL'].astype(str).str.upper().eq(model_name)]
        if not wrong.empty:
            raise ValueError(f'Parameter CSV contains rows not labelled {model_name}')
    parameters['TRADE_DT']=pd.to_datetime(parameters.TRADE_DT)
    parameters['EXPIRY']=pd.to_datetime(parameters.EXPIRY)
    if 'RISK_FREE_RATE' not in parameters.columns:
        rates = OPT.groupby(['TRADE_DT','EXPIRY'], as_index=False)[
            'RISK_FREE_RATE'
        ].first()
        parameters = parameters.merge(rates, on=['TRADE_DT','EXPIRY'], how='left')
        if parameters['RISK_FREE_RATE'].isna().any():
            raise ValueError('Unable to recover RISK_FREE_RATE from option panel')
    return parameters

OPT['TRADE_DT']=pd.to_datetime(OPT.TRADE_DT)
OPT['EXPIRY']=pd.to_datetime(OPT.EXPIRY)
PAR=load_volatility_parameters(VOLATILITY_MODEL)

def resolve_model_pair(pars: dict, start_date, end_date, expiry, end_row: DataRow) -> Tuple[DataRow,DataRow]:
    """Use the observed end-date curve, or carry the last curve to a terminal option date where IV fitting is unavailable."""
    p0=pars[(start_date,expiry)]
    if (end_date,expiry) in pars: return p0,pars[(end_date,expiry)]
    values=p0._asdict() if hasattr(p0,'_asdict') else dict(p0)
    p1=pd.Series(values,dtype=object); p1['TRADE_DT']=pd.Timestamp(end_date); p1['EXPIRY']=pd.Timestamp(expiry)
    p1['FORWARD']=float(end_row.FORWARD); p1['TAU']=float(end_row.TAU); p1['RISK_FREE_RATE']=float(end_row.RISK_FREE_RATE)
    return p0,p1


In [4]:
def surface_iv(model: DataRow, k: ScalarOrArray, tau: float | None = None) -> ScalarOrArray:
    """固定二次模型sigma(k)=a+b*k+0.5*c*k^2。"""
    x=np.asarray(k,dtype=float)
    return np.maximum(float(model.a)+float(model.b)*x+.5*float(model.c)*x*x,1e-6)

In [5]:
def factor_volatility(factors: np.ndarray, k: float, put_k: float=np.nan, call_k: float=np.nan) -> float:
    """由[ATM, Call Skew, Put Skew]严格还原二次IV。"""
    atm,call_skew,put_skew=map(float,factors); h=float(HALF_SKEW_H)
    call_weight=k/(2*h)+k*k/(2*h*h)
    put_weight=-k/(2*h)+k*k/(2*h*h)
    return max(atm+call_weight*call_skew+put_weight*put_skew,1e-6)


def fixed_anchor_factors(p0: DataRow,p1: DataRow,F0: float,F1: float,tau0: float,tau1: float):
    """在期初固定执行价锚点上构造期初/期末Half-Skew状态。"""
    h=float(HALF_SKEW_H); shift=math.log(F0/F1)
    old_vols=np.array([surface_iv(p0,0.,tau0),surface_iv(p0,h,tau0),surface_iv(p0,-h,tau0)],float)
    new_vols=np.array([surface_iv(p1,shift,tau1),surface_iv(p1,h+shift,tau1),surface_iv(p1,-h+shift,tau1)],float)
    old=np.array([old_vols[0],old_vols[1]-old_vols[0],old_vols[2]-old_vols[0]])
    new=np.array([new_vols[0],new_vols[1]-new_vols[0],new_vols[2]-new_vols[0]])
    return old,new,-h,h

## 3. Black–76 定价、风险与单腿归因

计算期权价值、组合风险和相邻交易日的模型全重估 PnL。\n\nBlack–76：$V=e^{-r\tau}[F N(d_1)-K N(d_2)]$（Put 使用对应负号形式）。


In [6]:
def price(F: float, K: float, v: float, t: float, r: float, cp: str) -> float:
    """使用 Black--76 公式计算期权价格。"""
    if t<=0 or v<=0: return max((F-K) if cp=='CALL' else (K-F),0)*math.exp(-r*max(t,0))
    z=(math.log(F/K)+.5*v*v*t)/(v*math.sqrt(t)); d=math.exp(-r*t)
    return d*((F*ndtr(z)-K*ndtr(z-v*math.sqrt(t))) if cp=='CALL' else (K*ndtr(-z+v*math.sqrt(t))-F*ndtr(-z)))


In [7]:
# 单只期权Half-Skew风险。
def risks(row: DataRow,p: DataRow) -> Tuple[np.ndarray,float,float]:
    F,K,t,r=map(float,[row.FORWARD,row.STRIKE,row.TAU,row.RISK_FREE_RATE]); cp=row.TYPE
    k=math.log(K/F); v=float(surface_iv(p,k,t)); z=(math.log(F/K)+.5*v*v*t)/(v*math.sqrt(t))
    disc=math.exp(-r*t); phi=math.exp(-z*z/2)/math.sqrt(2*math.pi)
    delta=disc*(ndtr(z) if cp=='CALL' else -ndtr(-z)); gamma=disc*phi/(F*v*math.sqrt(t)); vega=disc*F*phi*math.sqrt(t)
    h=float(HALF_SKEW_H)
    call_weight=k/(2*h)+k*k/(2*h*h); put_weight=-k/(2*h)+k*k/(2*h*h)
    atmvol=vega; call_skew=vega*call_weight; put_skew=vega*put_weight
    val=price(F,K,v,t,r,cp); dvdt=-r*val+disc*F*phi*v/(2*math.sqrt(t))
    d2=z-v*math.sqrt(t); atmvol_vanna=-disc*phi*d2/v
    return np.array([delta,gamma,atmvol,call_skew,put_skew,atmvol_vanna,dvdt]),v,k

In [8]:
def model_leg_pnl(q: float, row0: DataRow, row1: DataRow, p0: DataRow, p1: DataRow) -> float:
    """Exact fixed-anchor model repricing PnL."""
    F0,F1=float(row0.FORWARD),float(row1.FORWARD)
    fixed_k=math.log(float(row0.STRIKE)/F0)
    factors0,factors1,put_k,call_k=fixed_anchor_factors(
        p0,p1,F0,F1,float(row0.TAU),float(row1.TAU)
    )
    vol0=factor_volatility(factors0,fixed_k,put_k,call_k)
    vol1=factor_volatility(factors1,fixed_k,put_k,call_k)
    value0=price(F0,float(row0.STRIKE),vol0,float(row0.TAU),float(row0.RISK_FREE_RATE),row0.TYPE)
    value1=price(F1,float(row0.STRIKE),vol1,float(row1.TAU),float(row1.RISK_FREE_RATE),row0.TYPE)
    return float(q)*M*(value1-value0)


## 4. Six independent fixed-expiry backtests


In [9]:
def run(expiry_code: str, bound: int = 100, minvol: int = 200, kmax: float = .22, ridge: float = 1e-5, max_options: int = MAX_OPTIONS,
        use_premium_margin_filter: bool = bool(STRATEGY_USE_PREMIUM_MARGIN_FILTER),
        min_premium_margin_ratio: float = float(STRATEGY_MIN_PREMIUM_MARGIN_RATIO),
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """逐日筛选期限与 OTM 期权、优化整数仓位、期货对冲并计算 PnL。"""
    expiry_code=str(expiry_code)
    expiry_rows=OPT.loc[OPT.EXPIRY_CODE.astype(str).eq(expiry_code) & OPT.SOURCE.eq('FUTURE')]
    dates=sorted(d for d in expiry_rows.TRADE_DT.unique() if pd.Timestamp(d).month in (6,7)); pars={(x.TRADE_DT,x.EXPIRY):x for x in PAR.itertuples() if not str(x.FIT_STATUS).startswith(('FAILED','INSUFFICIENT'))}
    pos=[]; pnl=[]; moneyness_stats=[]
    previous_option_qty = {}
    previous_futures_qty = 0.0
    previous_futures_expiry = None
    last_futures_settlement_price = np.nan
    for di,d in enumerate(dates[:-1]):
        dn=dates[di+1]; today=OPT[OPT.TRADE_DT.eq(d)]; tomorrow=OPT[OPT.TRADE_DT.eq(dn)]
        target_table = (STRATEGY_HALF_SKEW_TARGETS_AFTER if pd.Timestamp(d) >= pd.Timestamp(STRATEGY_HALF_SKEW_TARGET_SWITCH_DATE) else STRATEGY_HALF_SKEW_TARGETS_BEFORE)
        target_call = float(target_table[expiry_code]['CALL_SKEW'])
        target_put = float(target_table[expiry_code]['PUT_SKEW'])
        exps=sorted(set(today.loc[today.EXPIRY_CODE.astype(str).eq(expiry_code),'EXPIRY']) & set(tomorrow.loc[tomorrow.EXPIRY_CODE.astype(str).eq(expiry_code),'EXPIRY']))
        exps=[e for e in exps if (d,e) in pars and (e-d).days >= int(STRATEGY_MIN_DTE_DAYS)]
        if not exps: continue
        best=None
        for e in exps:
            x=today[today.EXPIRY.eq(e)&today.VOLUME.ge(minvol)].merge(tomorrow[['CODE']],on='CODE')
            p0=pars[(d,e)]; rr=[]
            for row in x.itertuples():
                g,v,k=risks(row,p0)
                # OTM-only universe: put strikes at/below forward and call
                # strikes at/above forward.  ITM contracts are excluded.
                is_otm = ((row.TYPE == 'CALL' and row.STRIKE >= row.FORWARD)
                          or (row.TYPE == 'PUT' and row.STRIKE <= row.FORWARD))
                if not is_otm:
                    continue
                if use_premium_margin_filter:
                    call_otm = max(float(row.STRIKE) - float(row.SPOT), 0.0)
                    put_otm = max(float(row.SPOT) - float(row.STRIKE), 0.0)
                    otm_amount = M * (call_otm if row.TYPE == 'CALL' else put_otm)
                    premium_amount = M * float(row.PRICE)
                    margin_per_short = premium_amount + max(
                        OPTION_MARGIN_RATE * M * float(row.SPOT) - otm_amount,
                        MINIMUM_MARGIN_RATE * M * float(row.SPOT),
                    )
                    premium_margin_ratio = premium_amount / margin_per_short
                    if premium_margin_ratio < float(min_premium_margin_ratio):
                        continue
                if abs(k) <= kmax and v < float(STRATEGY_MAX_MODEL_IV): rr.append((row,g,k))
            rr=sorted(rr,key=lambda z:z[2])
            # Complete eligible universe before applying the selected-leg cap.
            eligible_rr=list(rr)
            fixed_atm=[]
            if bool(STRATEGY_INCLUDE_ATM_OPTIONS) and int(STRATEGY_ATM_STRIKES_PER_SIDE) > 0:
                per_side=int(STRATEGY_ATM_STRIKES_PER_SIDE)
                # Below Forward keep the nearest OTM puts; above Forward keep
                # the nearest OTM calls.  Strict k signs avoid duplicating an
                # exact-ATM strike across Call and Put.
                left=sorted((z for z in rr if z[0].TYPE=='PUT' and z[2] <= 0.0), key=lambda z:abs(z[2]))
                right=sorted((z for z in rr if z[0].TYPE=='CALL' and z[2] > 0.0), key=lambda z:abs(z[2]))
                def nearest_unique_strikes(items, count):
                    selected=[]; strikes=set()
                    for item in items:
                        strike=float(item[0].STRIKE)
                        if strike in strikes: continue
                        selected.append(item); strikes.add(strike)
                        if len(selected) >= count: break
                    return selected
                fixed_atm=nearest_unique_strikes(left,per_side)+nearest_unique_strikes(right,per_side)
            fixed_codes={z[0].CODE for z in fixed_atm}
            regular=[z for z in rr if z[0].CODE not in fixed_codes]
            # Fixed ATM options occupy part of the unchanged total leg cap.
            # The remaining slots retain the original broad k-ranked sample.
            regular_slots=max(0,int(max_options)-len(fixed_atm))
            if regular_slots and regular:
                ids=np.unique(np.linspace(0,len(regular)-1,min(regular_slots,len(regular))).astype(int))
                regular=[regular[i] for i in ids]
            else:
                regular=[]
            rr=sorted(fixed_atm+regular,key=lambda z:z[2])
            # Minimum count applies to the final combined set, so enabling
            # fixed ATM options does not accidentally tighten this filter.
            if len(rr) < int(STRATEGY_MIN_CANDIDATE_OPTIONS): continue
            # Rows: Gamma, ATMVol, Call Skew, Put Skew, ATMVol Vanna, V_tau.
            full_risk_matrix=np.column_stack([z[1][[1,2,3,4,5,6]] for z in rr])
            optimization_rows=[]; target_values=[]
            if STRATEGY_HEDGE_GAMMA: optimization_rows.append(0); target_values.append(0.0)
            if STRATEGY_HEDGE_ATM_VOL: optimization_rows.append(1); target_values.append(0.0)
            optimization_rows.extend([2,3]); target_values.extend([target_call,target_put])
            if STRATEGY_HEDGE_ATMVOL_VANNA: optimization_rows.append(4); target_values.append(0.0)
            if STRATEGY_HEDGE_THETA: optimization_rows.append(5); target_values.append(0.0)
            # Half-Skew目标沿用原06的单份Greek加总口径；合约乘数仅用于PnL与归因。
            A=full_risk_matrix[optimization_rows,:]
            target_vector=np.asarray(target_values,dtype=float)
            scales=np.maximum(np.linalg.norm(A,axis=1),1e-9); An=A/scales[:,None]; bn=target_vector/scales
            # Conservative per-contract capital proxy: use the larger of long premium
            # and standalone short margin.  Normalize by the cross-sectional median so
            # STRATEGY_CAPITAL_PENALTY is dimensionless and comparable across dates.
            capital_cost=[]
            for option_row,_,_ in rr:
                call_otm=max(float(option_row.STRIKE)-float(option_row.SPOT),0.0)
                put_otm=max(float(option_row.SPOT)-float(option_row.STRIKE),0.0)
                otm_amount=M*(call_otm if option_row.TYPE=='CALL' else put_otm)
                premium_amount=M*float(option_row.PRICE)
                short_margin=premium_amount+max(
                    OPTION_MARGIN_RATE*M*float(option_row.SPOT)-otm_amount,
                    MINIMUM_MARGIN_RATE*M*float(option_row.SPOT),
                )
                capital_cost.append(max(premium_amount,short_margin))
            capital_cost=np.asarray(capital_cost,dtype=float)
            capital_scale=max(float(np.median(capital_cost)),1e-9)
            capital_weights=capital_cost/capital_scale
            capital_penalty=float(STRATEGY_CAPITAL_PENALTY)
            if capital_penalty < 0:
                raise ValueError('STRATEGY_CAPITAL_PENALTY must be non-negative')
            regularizers=[math.sqrt(ridge)*np.eye(len(rr))]
            if capital_penalty > 0:
                regularizers.append(math.sqrt(capital_penalty)*np.diag(capital_weights))
            AA=np.vstack([An,*regularizers]); bb=np.r_[bn,np.zeros(sum(x.shape[0] for x in regularizers))]
            candidate_codes = [item[0].CODE for item in rr]
            trade_limit = float(STRATEGY_MAX_DAILY_TRADE_PER_LEG)
            previous = np.array([previous_option_qty.get(code, 0.0) for code in candidate_codes])
            # Asymmetric position bounds: reducing/closing an existing leg is
            # unlimited, while same-direction additions and reverse-direction
            # new openings are capped by trade_limit.  Examples with limit=30:
            # +100 may move anywhere in [-30,+100]; -100 anywhere in [-100,+30].
            lower = np.where(
                previous > 0.0,
                -trade_limit,
                np.maximum(-float(bound), previous - trade_limit),
            )
            upper = np.where(
                previous < 0.0,
                trade_limit,
                np.minimum(float(bound), previous + trade_limit),
            )
            sol=lsq_linear(AA,bb,bounds=(lower,upper),lsmr_tol='auto').x
            # Apply the configured option-lot convention.
            if STRATEGY_INTEGER_OPTION_POSITIONS:
                sol=np.clip(np.rint(sol), np.ceil(lower), np.floor(upper)).astype(int)
            best=(e,rr,sol,full_risk_matrix,eligible_rr)
        if best is None: continue
        e,rr,q,full_risk_matrix,eligible_rr=best
        selected_k=[float(z[2]) for z in rr]
        eligible_k=[float(z[2]) for z in eligible_rr]
        moneyness_record={'date':pd.Timestamp(d),'expiry':pd.Timestamp(e),'expiry_code':expiry_code}
        for selected_index in range(int(max_options)):
            moneyness_record[f'SELECTED_LOG_MONEYNESS_{selected_index+1}']=(selected_k[selected_index] if selected_index < len(selected_k) else np.nan)
        moneyness_record['ELIGIBLE_MIN_LOG_MONEYNESS']=min(eligible_k) if eligible_k else np.nan
        moneyness_record['ELIGIBLE_MAX_LOG_MONEYNESS']=max(eligible_k) if eligible_k else np.nan
        moneyness_record['MAX_ELIGIBLE_OPTION_COUNT']=len(eligible_k)
        moneyness_stats.append(moneyness_record)
        end_reference=tomorrow.loc[tomorrow.EXPIRY.eq(e)].iloc[0]
        p0,p1=resolve_model_pair(pars,d,dn,e,end_reference)
        # Convert option Delta (yuan per index point) into the configured IM position.
        option_delta=M*sum(q[j]*rr[j][1][0] for j in range(len(rr)))
        raw_fut=-option_delta/FUTURES_MULTIPLIER
        fut=int(round(raw_fut)) if STRATEGY_INTEGER_FUTURES_POSITIONS else float(raw_fut)
        current_option_qty = {
            rr[j][0].CODE: (int(q[j]) if STRATEGY_INTEGER_OPTION_POSITIONS else float(q[j])) for j in range(len(rr)) if q[j] != 0
        }
        traded_codes = set(previous_option_qty) | set(current_option_qty)
        option_trade_qty = sum(
            abs(current_option_qty.get(code, 0) - previous_option_qty.get(code, 0))
            for code in traded_codes
        )
        # Liquidation costs are booked once after the final *actual* holding period.
        option_fee = FEE * option_trade_qty
        previous_option_qty = current_option_qty
        model=market=0
        for j,(row,g,k) in enumerate(rr):
            qj=q[j]
            if abs(qj)<1e-7: continue
            row1=tomorrow[tomorrow.CODE.eq(row.CODE)].iloc[0]
            model+=model_leg_pnl(qj,row,pd.Series(row1),p0,p1); market+=qj*M*(row1.PRICE-row.PRICE)
            pos.append([d,row.CODE,e,'OPTION',(int(qj) if STRATEGY_INTEGER_OPTION_POSITIONS else float(qj)),k,row.VOLUME,row.PRICE,
                        row.TYPE,row.STRIKE,row.SPOT])
        # Futures hedge belongs entirely to pure DELTA.
        df=float(tomorrow[tomorrow.EXPIRY.eq(e)].FORWARD.iloc[0]-today[today.EXPIRY.eq(e)].FORWARD.iloc[0])
        fut_pnl=fut*FUTURES_MULTIPLIER*df
        current_futures_price = float(today[today.EXPIRY.eq(e)].FORWARD.iloc[0])
        next_futures_price = float(tomorrow[tomorrow.EXPIRY.eq(e)].FORWARD.iloc[0])
        if previous_futures_expiry == e:
            # Same contract: charge only the net adjustment, e.g. 10 -> 2 means 8 lots.
            futures_trade_notional = (
                abs(fut - previous_futures_qty)
                * current_futures_price * FUTURES_MULTIPLIER
            )
        else:
            # Contract changed: close the previous expiry and open the new expiry.
            previous_close_notional = 0.0
            if previous_futures_expiry is not None and abs(previous_futures_qty) > 0:
                previous_rows = today[today.EXPIRY.eq(previous_futures_expiry)]
                if previous_rows.empty:
                    raise ValueError(
                        f'Missing current forward for previous futures expiry: '
                        f'{previous_futures_expiry}'
                    )
                previous_close_notional = (
                    abs(previous_futures_qty)
                    * float(previous_rows.FORWARD.iloc[0]) * FUTURES_MULTIPLIER
                )
            futures_trade_notional = (
                previous_close_notional
                + abs(fut) * current_futures_price * FUTURES_MULTIPLIER
            )
        futures_opening_fee = futures_trade_notional * FUTURES_FEE_RATE
        futures_closing_fee = 0.0
        futures_fee = futures_opening_fee + futures_closing_fee
        fee=option_fee+futures_fee
        previous_futures_qty = fut
        last_futures_settlement_price = next_futures_price
        previous_futures_expiry = e
        model+=fut_pnl; market+=fut_pnl
        pos.append([d,f'IM_{pd.Timestamp(e):%Y%m%d}',e,'FUTURE',fut,0.0,np.nan,float(today[today.EXPIRY.eq(e)].FORWARD.iloc[0]),np.nan,np.nan,float(today[today.EXPIRY.eq(e)].SPOT.iloc[0])])
        gross=sum(abs(q))
        # 策略目标与每日exposure使用单份Greek加总口径。
        cash_risk_exposures = full_risk_matrix@q
        actual_call=float(cash_risk_exposures[2]); actual_put=float(cash_risk_exposures[3])
        call_error=actual_call-target_call; put_error=actual_put-target_put
        call_completion=actual_call/target_call if abs(target_call)>1e-12 else np.nan
        put_completion=actual_put/target_put if abs(target_put)>1e-12 else np.nan
        pnl.append([dn,e,expiry_code,model,market,market-model,option_fee,futures_opening_fee,futures_closing_fee,futures_fee,fee,market-fee,gross,fut,*cash_risk_exposures,target_call,target_put,call_error,put_error,call_completion,put_completion,actual_call-actual_put,actual_call+actual_put])
    # The final valid holding interval may end before the final source-data date (e.g. DTE or candidate filters).
    # Charge liquidation exactly once at that interval's t+1 settlement price.
    if pnl:
        final_option_closing_fee=FEE*sum(abs(qty) for qty in previous_option_qty.values())
        final_futures_closing_fee=(abs(previous_futures_qty)*last_futures_settlement_price*FUTURES_MULTIPLIER*FUTURES_FEE_RATE
                                   if np.isfinite(last_futures_settlement_price) else 0.0)
        final_fee=final_option_closing_fee+final_futures_closing_fee
        pnl[-1][6]+=final_option_closing_fee; pnl[-1][8]+=final_futures_closing_fee
        pnl[-1][9]+=final_futures_closing_fee; pnl[-1][10]+=final_fee; pnl[-1][11]-=final_fee
    cols=['date','expiry','expiry_code','MODEL_PNL','MARKET_PNL','MARKET_NOISE_PNL','OPTION_FEE','FUTURES_OPENING_FEE','FUTURES_CLOSING_FEE','FUTURES_FEE','FEE','ACTUAL_PNL','GROSS_OPTION_POSITION','FUTURES_POSITION','GAMMA_EXPOSURE','ATMVOL_EXPOSURE','CALL_SKEW_EXPOSURE','PUT_SKEW_EXPOSURE','ATMVOL_VANNA_EXPOSURE','THETA_EXPOSURE','TARGET_CALL_SKEW','TARGET_PUT_SKEW','CALL_SKEW_TARGET_ERROR','PUT_SKEW_TARGET_ERROR','CALL_SKEW_COMPLETION_RATIO','PUT_SKEW_COMPLETION_RATIO','DIRECTIONAL_SKEW_EXPOSURE','WING_CURVATURE_EXPOSURE']
    return (pd.DataFrame(pnl,columns=cols),
            pd.DataFrame(pos,columns=['date','code','expiry','asset_type','qty','k','volume','price','option_type','strike','spot']),
            pd.DataFrame(moneyness_stats))

## 5. 保证金

分别计算期权逐腿/逐日保证金与期货逐腿/逐日保证金。


In [10]:
# 保证金辅助函数。
def calculate_option_margin(positions: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """按交易所公式计算逐腿及逐日期权卖方保证金。"""
    option = positions.loc[positions['asset_type'].eq('OPTION')].copy()
    call_otm = np.maximum(option['strike'] - option['spot'], 0.0)
    put_otm = np.maximum(option['spot'] - option['strike'], 0.0)
    option['OTM_AMOUNT'] = np.where(
        option['option_type'].eq('CALL'), M * call_otm, M * put_otm
    )
    base = OPTION_MARGIN_RATE * M * option['spot'] - option['OTM_AMOUNT']
    minimum = MINIMUM_MARGIN_RATE * M * option['spot']
    option['MARGIN_PER_SHORT_CONTRACT'] = (
        M * option['price'] + np.maximum(base, minimum)
    )
    option['SHORT_CONTRACTS'] = np.maximum(-option['qty'], 0).astype(int)
    option['LONG_CONTRACTS'] = np.maximum(option['qty'], 0).astype(int)
    option['LONG_OPTION_PREMIUM'] = (
        option['LONG_CONTRACTS'] * option['price'] * M
    )
    option['POSITION_MARGIN'] = (
        option['SHORT_CONTRACTS'] * option['MARGIN_PER_SHORT_CONTRACT']
    )
    daily = option.groupby('date', as_index=False).agg(
        OPTION_MARGIN=('POSITION_MARGIN', 'sum'),
        SHORT_OPTION_CONTRACTS=('SHORT_CONTRACTS', 'sum'),
        LONG_OPTION_CONTRACTS=('LONG_CONTRACTS', 'sum'),
        LONG_OPTION_PREMIUM=('LONG_OPTION_PREMIUM', 'sum'),
    )
    return option, daily


In [11]:
def calculate_futures_margin(positions: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """按期货手数、乘数和保证金率计算逐腿及逐日保证金。"""
    futures = positions.loc[positions['asset_type'].eq('FUTURE')].copy()
    futures['FUTURES_CONTRACTS'] = futures['qty'].astype(int)
    futures['FUTURES_MULTIPLIER'] = FUTURES_MULTIPLIER
    futures['FUTURES_MARGIN_RATE'] = FUTURES_MARGIN_RATE
    futures['POSITION_MARGIN'] = (
        futures['FUTURES_CONTRACTS'].abs()
        * futures['price'] * FUTURES_MULTIPLIER * FUTURES_MARGIN_RATE
    )
    daily = futures.groupby('date', as_index=False).agg(
        FUTURES_MARGIN=('POSITION_MARGIN', 'sum'),
        FUTURES_CONTRACTS=('FUTURES_CONTRACTS', 'sum'),
        ABS_FUTURES_CONTRACTS=('FUTURES_CONTRACTS', lambda x: int(x.abs().sum())),
    )
    return futures, daily


## 6. 图形与持仓诊断

输出累计 PnL 图以及所选到期日的 Skew 期限段诊断。


In [12]:
def plot_cumulative_pnl(pnl: pd.DataFrame, output_dir: Path | str) -> pd.DataFrame:
    """Generate cumulative and daily account PnL charts for one fixed expiry."""
    output_dir=Path(output_dir); overview_dir=output_dir/'overview'; component_dir=output_dir/'components'
    overview_dir.mkdir(parents=True,exist_ok=True); component_dir.mkdir(parents=True,exist_ok=True)
    data=pnl.copy(); data['date']=pd.to_datetime(data.date)
    columns=['ACTUAL_PNL','MARKET_PNL','MODEL_PNL','MARKET_NOISE_PNL']
    cumulative=data.set_index('date')[columns].cumsum()
    fig,ax=plt.subplots(figsize=(13,6.5))
    for column,color,width in [('ACTUAL_PNL','#102a43',2.6),('MARKET_PNL','#2f80ed',1.8),('MODEL_PNL','#27ae60',1.6)]:
        ax.plot(cumulative.index,cumulative[column],label=column,color=color,linewidth=width)
    ax.axhline(0,color='#667085',linestyle='--',linewidth=.9); ax.set_title('Cumulative Total PnL')
    ax.set_xlabel('Date'); ax.set_ylabel('Cumulative PnL'); ax.grid(alpha=.3); ax.legend(); ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    fig.autofmt_xdate(); fig.tight_layout(); fig.savefig(overview_dir/'01_cumulative_total_pnl.png',dpi=180,bbox_inches='tight'); plt.close(fig)
    fig,ax=plt.subplots(figsize=(13,6.5))
    for column,color,width in [('ACTUAL_PNL','#102a43',2.4),('MARKET_PNL','#2f80ed',1.5),('MODEL_PNL','#27ae60',1.4)]:
        ax.plot(data.date,data[column],label=column,color=color,linewidth=width)
    ax.axhline(0,color='#667085',linestyle='--',linewidth=.9); ax.set_title('Daily PnL')
    ax.set_xlabel('Date'); ax.set_ylabel('Daily PnL'); ax.grid(alpha=.3); ax.legend(); ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    fig.autofmt_xdate(); fig.tight_layout(); fig.savefig(overview_dir/'02_daily_pnl.png',dpi=180,bbox_inches='tight'); plt.close(fig)
    for column in columns:
        fig,ax=plt.subplots(figsize=(10.5,5)); ax.plot(cumulative.index,cumulative[column],linewidth=2,color='#3568b8')
        ax.axhline(0,color='#667085',linestyle='--',linewidth=.8); ax.set_title(f'Cumulative {column}'); ax.set_xlabel('Date'); ax.set_ylabel('Cumulative PnL'); ax.grid(alpha=.3)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d')); fig.autofmt_xdate(); fig.tight_layout(); fig.savefig(component_dir/f'{column.lower()}_cumulative_pnl.png',dpi=180,bbox_inches='tight'); plt.close(fig)
    cumulative.reset_index().to_csv(output_dir/'cumulative_pnl.csv',index=False)
    return cumulative


In [13]:
def plot_selected_expiry_half_skew_segments(positions: pd.DataFrame, model_parameters: pd.DataFrame, output_dir: Path | str) -> pd.DataFrame:
    """绘制每个持有期所选到期日的跨日 Skew 变化线段。"""
    if plt is None:
        raise RuntimeError('matplotlib is required to generate Skew segment chart')
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    pos = positions.loc[positions['asset_type'].eq('OPTION')].copy()
    pos['date'] = pd.to_datetime(pos['date']).dt.normalize()
    pos['expiry'] = pd.to_datetime(pos['expiry']).dt.normalize()
    selected = pos.groupby('date', as_index=False)['expiry'].first().sort_values('date')
    parameters = model_parameters.copy()
    parameters['TRADE_DT'] = pd.to_datetime(parameters['TRADE_DT']).dt.normalize()
    parameters['EXPIRY'] = pd.to_datetime(parameters['EXPIRY']).dt.normalize()
    lookup = {(row.TRADE_DT, row.EXPIRY): row for row in parameters.itertuples(index=False)}
    market_dates = sorted(parameters['TRADE_DT'].unique())
    next_date = {pd.Timestamp(market_dates[i]): pd.Timestamp(market_dates[i + 1])
                 for i in range(len(market_dates) - 1)}
    rows = []
    for row in selected.itertuples(index=False):
        start_date, expiry = pd.Timestamp(row.date), pd.Timestamp(row.expiry)
        end_date = next_date.get(start_date)
        if end_date is None or (start_date, expiry) not in lookup or (end_date, expiry) not in lookup:
            continue
        start_model, end_model = lookup[(start_date, expiry)], lookup[(end_date, expiry)]
        start_factors,end_factors,_,_=fixed_anchor_factors(start_model,end_model,float(start_model.FORWARD),float(end_model.FORWARD),float(start_model.TAU),float(end_model.TAU))
        rows.append({
            'START_DATE': start_date, 'END_DATE': end_date, 'EXPIRY': expiry,
            'EXPIRY_CODE': f'{expiry.year % 100:02d}{expiry.month:02d}',
                        'CALL_SKEW_START':float(start_factors[1]),'CALL_SKEW_END':float(end_factors[1]),
            'CALL_SKEW_CHANGE':float(end_factors[1]-start_factors[1]),
            'PUT_SKEW_START':float(start_factors[2]),'PUT_SKEW_END':float(end_factors[2]),
            'PUT_SKEW_CHANGE':float(end_factors[2]-start_factors[2]),
        })
    segments = pd.DataFrame(rows)
    segments.to_csv(output_dir/'selected_expiry_half_skew_segments.csv', index=False, date_format='%Y-%m-%d')

    if segments.empty:
        warnings.warn('No fixed-anchor Half-Skew segments available; chart skipped',RuntimeWarning)
        return segments

    codes = sorted(segments['EXPIRY_CODE'].unique())
    cmap = plt.get_cmap('tab10')
    colors = {code: cmap(i % 10) for i, code in enumerate(codes)}

    def plot_half_skews(filename: str) -> Path:
        fig, ax = plt.subplots(figsize=(15, 7.5))
        factor_specs = [
            ('CALL_SKEW_START', 'CALL_SKEW_END', 'Call Skew', '#2b6cb0'),
            ('PUT_SKEW_START', 'PUT_SKEW_END', 'Put Skew', '#d1495b'),
        ]
        for start_column, end_column, label, color in factor_specs:
            labelled = False
            for row in segments.itertuples(index=False):
                ax.plot(
                    [row.START_DATE, row.END_DATE],
                    [getattr(row, start_column), getattr(row, end_column)],
                    marker='o', markersize=4.2, linewidth=2.0,
                    color=color, alpha=.9, label=label if not labelled else None,
                )
                labelled = True
        ax.axhline(0.0, color='#667085', linestyle='--', linewidth=.9)
        ax.set_title(
            'Selected-Expiry Fixed-Anchor Call and Put Skew: One Segment per Holding Period',
            fontsize=13, fontweight='bold',
        )
        ax.set_xlabel('Trade Date')
        ax.set_ylabel('Fixed-anchor half skew')
        ax.yaxis.set_major_formatter(PercentFormatter(1.0))
        ax.grid(color='#d9e0e8', linewidth=.7, alpha=.8)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
        ax.legend(frameon=True, title='Factor')
        fig.autofmt_xdate(); fig.tight_layout()
        output_file = output_dir / filename
        fig.savefig(output_file, dpi=200, bbox_inches='tight')
        plt.close(fig)
        return output_file

    plot_half_skews('selected_expiry_call_put_skew_segments.png')
    return segments

# 全项目统一使用下一节定义的五因子状态；固定利率不参与 PnL 归因。

## 7. Complete second-order Taylor attribution


In [14]:
TAYLOR_FACTOR_NAMES = ['DELTA','ATM','CALL_SKEW','PUT_SKEW','TAU']


def taylor_state_vectors(
    row: DataRow, p0: DataRow, p1: DataRow,
) -> Tuple[np.ndarray, np.ndarray]:
    """构造前一期固定锚点的五因子期初与期末状态；利率固定且不参与归因。"""
    F0, F1 = float(row.FORWARD), float(p1.FORWARD)
    factors0,factors1,_,_=fixed_anchor_factors(p0,p1,F0,F1,float(row.TAU),float(p1.TAU))
    x0=np.r_[F0,factors0,float(row.TAU)]
    x1=np.r_[F1,factors1,float(p1.TAU)]
    return x0, x1


def taylor_price_from_state(row: DataRow, state: np.ndarray, p0: DataRow) -> float:
    """在期初固定曲面坐标上按五因子状态定价，使用项目固定利率。"""
    F_price,atm,call_skew,put_skew,tau=map(float,state)
    rate=float(row.RISK_FREE_RATE)
    fixed_k=math.log(float(row.STRIKE)/float(row.FORWARD))
    volatility=factor_volatility(np.array([atm,call_skew,put_skew]),fixed_k)
    return price(F_price, float(row.STRIKE), volatility, tau, rate, row.TYPE)


def numerical_taylor_exposures(
    row: DataRow, p0: DataRow, p1: DataRow, relative_step: float = 1e-4,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """用实际状态单位的中心差分计算五个一阶和完整 5x5 Hessian exposure。"""
    x0, x1 = taylor_state_vectors(row, p0, p1)
    # 每个状态使用与自身尺度相适应的 bump；这是求导步长，不是实际市场变化。
    floors=np.array([1.,1e-3,1e-3,1e-2,1/3650])
    bumps = relative_step*np.maximum(np.abs(x0), floors)
    base = taylor_price_from_state(row, x0, p0)
    gradient = np.zeros(len(x0)); hessian = np.zeros((len(x0), len(x0)))
    for i in range(len(x0)):
        ei = np.zeros(len(x0)); ei[i] = bumps[i]
        fp = taylor_price_from_state(row, x0 + ei, p0)
        fm = taylor_price_from_state(row, x0 - ei, p0)
        gradient[i] = (fp-fm)/(2*bumps[i])
        hessian[i, i] = (fp-2*base+fm)/(bumps[i]**2)
    for i in range(len(x0)):
        for j in range(i+1, len(x0)):
            ei = np.zeros(len(x0)); ej = np.zeros(len(x0))
            ei[i] = bumps[i]; ej[j] = bumps[j]
            cross = (
                taylor_price_from_state(row, x0+ei+ej, p0)
                - taylor_price_from_state(row, x0+ei-ej, p0)
                - taylor_price_from_state(row, x0-ei+ej, p0)
                + taylor_price_from_state(row, x0-ei-ej, p0)
            )/(4*bumps[i]*bumps[j])
            hessian[i, j] = hessian[j, i] = cross
    return gradient, hessian, x0, x1

In [15]:
def taylor_leg(
    q: float, row: DataRow, p0: DataRow, p1: DataRow, h: float = 1e-2,
) -> Tuple[np.ndarray, np.ndarray, float, Dict[str, float], Dict[str, float]]:
    """计算单腿五因子 Taylor PnL，并返回原始 exposure。"""
    gradient, hessian, x0, x1 = numerical_taylor_exposures(
        row, p0, p1, relative_step=h,
    )
    changes = x1-x0; scale = float(q)*M
    first_pnl = scale*gradient*changes
    allocated_second = first_pnl.copy()
    pnl_terms: Dict[str, float] = {}
    exposures: Dict[str, float] = {}
    for i, name in enumerate(TAYLOR_FACTOR_NAMES):
        pnl_terms[f'FIRST_{name}'] = first_pnl[i]
        pnl_terms[f'SECOND_{name}_{name}'] = .5*scale*hessian[i, i]*changes[i]**2
        exposures[f'FIRST_{name}_EXPOSURE'] = scale*gradient[i]
        exposures[f'SECOND_{name}_{name}_EXPOSURE'] = scale*hessian[i, i]
        allocated_second[i] += .5*scale*hessian[i, i]*changes[i]**2
    for i, left in enumerate(TAYLOR_FACTOR_NAMES):
        for j in range(i+1, len(TAYLOR_FACTOR_NAMES)):
            right = TAYLOR_FACTOR_NAMES[j]
            cross_pnl = scale*hessian[i, j]*changes[i]*changes[j]
            pnl_terms[f'CROSS_{left}_{right}'] = cross_pnl
            exposures[f'CROSS_{left}_{right}_EXPOSURE'] = scale*hessian[i, j]
            allocated_second[i] += .5*cross_pnl
            allocated_second[j] += .5*cross_pnl
    exact = scale*(taylor_price_from_state(row, x1, p0)-taylor_price_from_state(row, x0, p0))
    return first_pnl, allocated_second, exact, pnl_terms, exposures


def plot_individual_raw_exposures(
    exposure_data: pd.DataFrame, exposure_columns: list[str], output_dir: Path,
    title_prefix: str,
) -> None:
    """将每个 model Greek exposure 的原始值分别绘制为独立时间序列图。"""
    output_dir = Path(output_dir); output_dir.mkdir(parents=True, exist_ok=True)
    dates = pd.to_datetime(exposure_data['date'])
    for column in exposure_columns:
        figure, axis = plt.subplots(figsize=(13, 6.5))
        axis.plot(dates, exposure_data[column], linewidth=2, color='#1565c0')
        axis.axhline(0, color='#667085', linestyle='--', linewidth=.8)
        axis.set_title(f'{title_prefix}: {column}', fontweight='bold')
        axis.set_xlabel('Date'); axis.set_ylabel(f'Raw Model Greek Exposure: {column}')
        axis.grid(alpha=.3); axis.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
        figure.autofmt_xdate(); figure.tight_layout()
        filename = column.lower()+'.png'
        figure.savefig(output_dir/filename, dpi=200, bbox_inches='tight')
        plt.close(figure)


In [16]:
def calculate_t2_attribution(
    positions: pd.DataFrame, pnl_daily: pd.DataFrame, output_dir: Path | str, h: float=1e-4,
) -> Tuple[pd.DataFrame,pd.DataFrame,pd.DataFrame]:
    """Calculate only the complete second-order Taylor attribution and full Hessian exposures."""
    positions=positions.copy(); positions['date']=pd.to_datetime(positions.date); positions['expiry']=pd.to_datetime(positions.expiry)
    pnl_daily=pnl_daily.copy(); pnl_daily['date']=pd.to_datetime(pnl_daily.date)
    pars={(x.TRADE_DT,x.EXPIRY):x for x in PAR.itertuples() if not str(x.FIT_STATUS).startswith(('FAILED','INSUFFICIENT'))}
    records=[]; exposure_records=[]
    for date,all_positions in positions.groupby('date'):
        date=pd.Timestamp(date); option_positions=all_positions.loc[all_positions.asset_type.eq('OPTION')]
        later=pnl_daily.loc[pnl_daily.date.gt(date),'date']
        if option_positions.empty or later.empty: continue
        end_date=pd.Timestamp(later.min()); expiry=pd.Timestamp(option_positions.expiry.iloc[0])
        if (date,expiry) not in pars: continue
        old=OPT.loc[OPT.TRADE_DT.eq(date)&OPT.EXPIRY.eq(expiry)].set_index('CODE')
        new=OPT.loc[OPT.TRADE_DT.eq(end_date)&OPT.EXPIRY.eq(expiry)].set_index('CODE')
        p0,p1=resolve_model_pair(pars,date,end_date,expiry,new.iloc[0])
        second=np.zeros(len(TAYLOR_FACTOR_NAMES)); terms={}; exposures={}
        for leg in option_positions.itertuples(index=False):
            _,leg_second,_,leg_terms,leg_exposures=taylor_leg(leg.qty,old.loc[leg.code],p0,p1,h=h)
            second+=leg_second
            for name,value in leg_terms.items(): terms[name]=terms.get(name,0.)+value
            for name,value in leg_exposures.items(): exposures[name]=exposures.get(name,0.)+value
        F0=float(old.FORWARD.iloc[0]); F1=float(p1.FORWARD)
        futures_contracts=float(all_positions.loc[all_positions.asset_type.eq('FUTURE'),'qty'].iloc[0])
        futures_exposure=futures_contracts*FUTURES_MULTIPLIER; futures_pnl=futures_exposure*(F1-F0)
        second[0]+=futures_pnl; terms['FIRST_DELTA']=terms.get('FIRST_DELTA',0.)+futures_pnl; exposures['FIRST_DELTA_EXPOSURE']=exposures.get('FIRST_DELTA_EXPOSURE',0.)+futures_exposure
        realized=pnl_daily.loc[pnl_daily.date.eq(end_date)].iloc[0]; exact=float(realized.MODEL_PNL); fee_pnl=-float(realized.FEE)
        factor_sum=float(second.sum()); model_residual=exact-factor_sum; total_residual=float(realized.ACTUAL_PNL)-factor_sum-fee_pnl
        record={'date':end_date,'MODEL_PNL':exact,'MARKET_PNL':float(realized.MARKET_PNL),'ACTUAL_PNL_AFTER_FEES':float(realized.ACTUAL_PNL),'FEE_PNL':fee_pnl,**terms}
        for i,name in enumerate(TAYLOR_FACTOR_NAMES): record[f'T2_{name}']=second[i]
        record.update({'T2_MODEL_RESIDUAL':model_residual,'T2_TOTAL_RESIDUAL':total_residual,'T2_MARKET_NOISE':total_residual-model_residual})
        records.append(record); exposure_records.append({'date':date,'expiry':expiry,**exposures})
    result=pd.DataFrame(records); full_exposures=pd.DataFrame(exposure_records); output_dir=Path(output_dir)
    result.to_csv(output_dir/'t2_attribution.csv',index=False); full_exposures.to_csv(output_dir/'full_t2_exposures.csv',index=False)
    exposure_columns=[c for c in full_exposures.columns if c.endswith('_EXPOSURE')]
    plot_individual_raw_exposures(full_exposures,exposure_columns,output_dir/'figures'/'t2'/'exposures','T2 Raw Model Greek Exposure')
    term_columns=[c for c in result.columns if c.startswith(('FIRST_','SECOND_','CROSS_'))]
    values=[*term_columns,'MODEL_PNL','MARKET_PNL','ACTUAL_PNL_AFTER_FEES','FEE_PNL','T2_MODEL_RESIDUAL','T2_TOTAL_RESIDUAL','T2_MARKET_NOISE']
    complete=pd.DataFrame({'date':result.date})
    for column in values: complete[f'DAILY_{column}']=result[column]; complete[f'CUMULATIVE_{column}']=result[column].cumsum()
    complete.to_csv(output_dir/'complete_t2_pnl_daily_and_cumulative.csv',index=False)
    t2_skew_absolute=float((result.T2_CALL_SKEW+result.T2_PUT_SKEW).abs().sum())
    t2_non_skew_absolute={name:float(result[f'T2_{name}'].abs().sum()) for name in TAYLOR_FACTOR_NAMES if name not in {'CALL_SKEW','PUT_SKEW'}}
    t2_denominator=t2_skew_absolute+sum(t2_non_skew_absolute.values())
    t2_ranked_non_skew=sorted(t2_non_skew_absolute.items(),key=lambda item:item[1],reverse=True)
    summary_record={'method':'T2','signed_model_residual':result.T2_MODEL_RESIDUAL.sum(),'absolute_model_residual':result.T2_MODEL_RESIDUAL.abs().sum(),'half_skew_abs_share':t2_skew_absolute/t2_denominator if t2_denominator else np.nan,'call_skew_pnl':result.T2_CALL_SKEW.sum(),'put_skew_pnl':result.T2_PUT_SKEW.sum()}
    for rank,(name,value) in enumerate(t2_ranked_non_skew[:3],start=1):
        summary_record[f'top_{rank}_non_skew_factor']=name
        summary_record[f'top_{rank}_non_skew_abs_pnl']=value
        summary_record[f'top_{rank}_non_skew_share']=value/t2_denominator if t2_denominator else np.nan
    summary=pd.DataFrame([summary_record])
    summary.to_csv(output_dir/'t2_summary.csv',index=False)
    fig,ax=plt.subplots(figsize=(14,7))
    for name in TAYLOR_FACTOR_NAMES: ax.plot(pd.to_datetime(result.date),result[f'T2_{name}'].cumsum(),label=name,linewidth=2.3 if name in {'CALL_SKEW','PUT_SKEW'} else 1.3)
    ax.plot(pd.to_datetime(result.date),result.MODEL_PNL.cumsum(),label='MODEL PNL',color='black',linewidth=2.5)
    ax.axhline(0,color='#667085',linestyle='--',linewidth=.8); ax.set_title('Complete T2 Cumulative PnL Attribution'); ax.set_xlabel('Date'); ax.set_ylabel('Cumulative PnL'); ax.grid(alpha=.3); ax.legend(ncol=3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d')); fig.autofmt_xdate(); fig.tight_layout(); (output_dir/'figures'/'t2').mkdir(parents=True,exist_ok=True); fig.savefig(output_dir/'figures'/'t2'/'t2_cumulative_attribution.png',dpi=200,bbox_inches='tight'); plt.close(fig)
    return result,summary,full_exposures

## 8. Traditional Taylor：固定前期锚点的专用 Exposure

Traditional Delta 保持与实际期货对冲一致：

\[
\Pi_{\Delta}=\left(M\sum_jq_j\Delta^{Black}_{F,j}+N_tM_F\right)\Delta F.
\]

Traditional exposure 表仅包含本归因实际使用的导数：Delta、ATM、Skew、Curvature/BF25、Theta、Gamma、ATM Vanna 与 ATM Volga。


In [17]:
TRADITIONAL_TAYLOR_COMPONENTS = [
    'DELTA_PNL','GAMMA_PNL','ATMVOL_PNL','CALL_SKEW_PNL','PUT_SKEW_PNL',
    'THETA_PNL','ATMVOL_VANNA_PNL','ATMVOL_VOLGA_PNL',
]

TRADITIONAL_NON_SKEW_LABELS = {
    'DELTA_PNL':'Delta','GAMMA_PNL':'Gamma','ATMVOL_PNL':'ATM Vol',
    'THETA_PNL':'Theta','ATMVOL_VANNA_PNL':'ATM Vol Vanna',
    'ATMVOL_VOLGA_PNL':'ATM Vol Volga',
}


def traditional_absolute_attribution_breakdown(attribution: pd.DataFrame) -> Tuple[float, float, list]:
    """按日合并两侧 Skew 后，计算一致口径的绝对归因和非 Skew 排名。"""
    skew_absolute=float((attribution['CALL_SKEW_PNL']+attribution['PUT_SKEW_PNL']).abs().sum())
    non_skew_absolute={
        label:float(attribution[column].abs().sum())
        for column,label in TRADITIONAL_NON_SKEW_LABELS.items()
    }
    denominator=skew_absolute+sum(non_skew_absolute.values())
    ranked=sorted(non_skew_absolute.items(),key=lambda item:item[1],reverse=True)
    ranked=[(name,value,value/denominator if denominator else np.nan) for name,value in ranked]
    return skew_absolute,denominator,ranked



def traditional_greeks(row: DataRow, model: DataRow) -> Dict[str, float]:
    """计算固定前期锚点的 Traditional Taylor exposure。"""
    F,K,tau,r = map(float,[row.FORWARD,row.STRIKE,row.TAU,row.RISK_FREE_RATE])
    k=math.log(K/F); vol=float(surface_iv(model,k,tau)); sqrt_tau=math.sqrt(tau)
    d1=(math.log(F/K)+.5*vol*vol*tau)/(vol*sqrt_tau); d2=d1-vol*sqrt_tau
    discount=math.exp(-r*tau); phi=math.exp(-.5*d1*d1)/math.sqrt(2*math.pi)
    value=price(F,K,vol,tau,r,row.TYPE)
    delta=discount*(ndtr(d1) if row.TYPE=='CALL' else -ndtr(-d1))
    gamma=discount*phi/(F*vol*sqrt_tau); vega=discount*F*phi*sqrt_tau
    risk_vector,_,_=risks(row,model)
    return {
        'delta':delta,'gamma':gamma,'vega':risk_vector[2],'call_skew':risk_vector[3],'put_skew':risk_vector[4],
        'theta_tau':-r*value+discount*F*phi*vol/(2*sqrt_tau),
        'atmvol_vanna':-discount*phi*d2/vol,'atmvol_volga':vega*d1*d2/vol,
    }

In [18]:
def calculate_traditional_taylor_attribution(
    positions: pd.DataFrame, pnl_daily: pd.DataFrame, output_dir: Path | str,
    capital_daily: pd.DataFrame | None = None,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """计算固定前期锚点的 Traditional PnL 并输出 exposure。"""
    positions=positions.copy(); positions['date']=pd.to_datetime(positions.date)
    positions['expiry']=pd.to_datetime(positions.expiry)
    pnl_daily=pnl_daily.copy(); pnl_daily['date']=pd.to_datetime(pnl_daily.date)
    pars={(x.TRADE_DT,x.EXPIRY):x for x in PAR.itertuples()
          if not str(x.FIT_STATUS).startswith(('FAILED','INSUFFICIENT'))}
    records=[]; exposure_records=[]
    exposure_map={
        'DELTA_EXPOSURE':'delta',
        'GAMMA_EXPOSURE':'gamma','ATMVOL_EXPOSURE':'vega','CALL_SKEW_EXPOSURE':'call_skew',
        'PUT_SKEW_EXPOSURE':'put_skew','THETA_EXPOSURE':'theta_tau',
        'ATMVOL_VANNA_EXPOSURE':'atmvol_vanna','ATMVOL_VOLGA_EXPOSURE':'atmvol_volga'}

    for date, all_positions in positions.groupby('date'):
        date=pd.Timestamp(date); option_positions=all_positions[all_positions.asset_type.eq('OPTION')]
        later=pnl_daily.loc[pnl_daily.date.gt(date),'date']
        if option_positions.empty or later.empty: continue
        end_date=pd.Timestamp(later.min()); expiry=pd.Timestamp(option_positions.expiry.iloc[0])
        old=OPT[(OPT.TRADE_DT.eq(date))&OPT.EXPIRY.eq(expiry)].set_index('CODE')
        new=OPT[(OPT.TRADE_DT.eq(end_date))&OPT.EXPIRY.eq(expiry)].set_index('CODE')
        p0,p1=resolve_model_pair(pars,date,end_date,expiry,new.iloc[0])
        components={name:0.0 for name in TRADITIONAL_TAYLOR_COMPONENTS}
        exposures={name:0.0 for name in exposure_map}
        for leg in option_positions.itertuples(index=False):
            row0,row1=old.loc[leg.code],new.loc[leg.code]; g=traditional_greeks(row0,p0)
            dF=float(row1.FORWARD-row0.FORWARD)
            f0,f1,_,_=fixed_anchor_factors(p0,p1,float(row0.FORWARD),float(row1.FORWARD),float(row0.TAU),float(row1.TAU))
            da,db,dc=(f1-f0).astype(float)
            dtau=float(row1.TAU-row0.TAU); scale=float(leg.qty)*M
            components['DELTA_PNL']+=scale*g['delta']*dF
            components['GAMMA_PNL']+=.5*scale*g['gamma']*dF*dF
            components['ATMVOL_PNL']+=scale*g['vega']*da; components['CALL_SKEW_PNL']+=scale*g['call_skew']*db
            components['PUT_SKEW_PNL']+=scale*g['put_skew']*dc
            components['THETA_PNL']+=scale*g['theta_tau']*dtau
            components['ATMVOL_VANNA_PNL']+=scale*g['atmvol_vanna']*dF*da
            components['ATMVOL_VOLGA_PNL']+=.5*scale*g['atmvol_volga']*da*da
            for output_name, greek_name in exposure_map.items(): exposures[output_name]+=scale*g[greek_name]
        futures_contracts=float(all_positions.loc[all_positions.asset_type.eq('FUTURE'),'qty'].iloc[0])
        dF=float(new.FORWARD.iloc[0]-old.FORWARD.iloc[0]); futures_exposure=futures_contracts*FUTURES_MULTIPLIER
        components['DELTA_PNL']+=futures_exposure*dF; exposures['DELTA_EXPOSURE']+=futures_exposure
        realized=pnl_daily.loc[pnl_daily.date.eq(end_date)].iloc[0]; exact=float(realized.MODEL_PNL)
        greek_sum=sum(components.values()); fee_pnl=-float(realized.FEE)
        model_residual=exact-greek_sum; total_residual=float(realized.ACTUAL_PNL)-greek_sum-fee_pnl
        records.append({'date':end_date,**components,'MODEL_RESIDUAL':model_residual,
                        'MARKET_NOISE':total_residual-model_residual,'TOTAL_RESIDUAL':total_residual,
                        'FEE_PNL':fee_pnl,'MODEL_PNL':exact,'MARKET_PNL':float(realized.MARKET_PNL),
                        'ACTUAL_PNL_AFTER_FEES':float(realized.ACTUAL_PNL)})
        exposure_records.append({'date':date,'expiry':expiry,**exposures})
    attribution=pd.DataFrame(records); traditional_exposures=pd.DataFrame(exposure_records); output_dir=Path(output_dir)
    attribution.to_csv(output_dir/'traditional_taylor_attribution.csv',index=False)
    traditional_exposures.to_csv(output_dir/'traditional_taylor_exposures.csv',index=False)
    exposure_columns=[c for c in traditional_exposures.columns if c.endswith('_EXPOSURE')]
    figure_dir=output_dir/'figures'/'traditional_taylor'; figure_dir.mkdir(parents=True,exist_ok=True)
    plot_individual_raw_exposures(
        traditional_exposures, exposure_columns, figure_dir/'exposures',
        'Traditional Taylor Raw Model Greek Exposure',
    )
    skew_absolute,denominator,ranked_non_skew=traditional_absolute_attribution_breakdown(attribution)
    summary_record={'method':'TRADITIONAL_TAYLOR',
        'signed_model_residual':attribution.MODEL_RESIDUAL.sum(),
        'absolute_model_residual':attribution.MODEL_RESIDUAL.abs().sum(),
        'half_skew_abs_share':skew_absolute/denominator if denominator else np.nan,
        'call_skew_pnl':attribution.CALL_SKEW_PNL.sum(),'put_skew_pnl':attribution.PUT_SKEW_PNL.sum()}
    for rank,(name,value,share) in enumerate(ranked_non_skew[:3],start=1):
        summary_record[f'top_{rank}_non_skew_factor']=name
        summary_record[f'top_{rank}_non_skew_abs_pnl']=value
        summary_record[f'top_{rank}_non_skew_share']=share
    summary=pd.DataFrame([summary_record])
    summary.to_csv(output_dir/'traditional_taylor_summary.csv',index=False)
    # Compact 11x7 canvas fits the left side of a 16:9 slide and leaves room for notes.
    attribution['date']=pd.to_datetime(attribution.date); fig,ax=plt.subplots(figsize=(11,7))
    for component in TRADITIONAL_TAYLOR_COMPONENTS:
        ax.plot(attribution.date,attribution[component].cumsum(),label=component,
                linewidth=2.3 if component in {'CALL_SKEW_PNL','PUT_SKEW_PNL'} else 1.2)
    for column,label,style,color,width in [
        ('MODEL_RESIDUAL','MODEL RESIDUAL','--','black',2),('MARKET_NOISE','MARKET NOISE',':','#c0392b',1.7),
        ('TOTAL_RESIDUAL','TOTAL RESIDUAL','-.','#8e44ad',1.8),('FEE_PNL','FEE PNL',':','#795548',1.5),
        ('MODEL_PNL','MODEL PNL','-','#34495e',2.6),('MARKET_PNL','MARKET PNL','-','#1565c0',2.6),
        ('ACTUAL_PNL_AFTER_FEES','ACTUAL PNL AFTER FEES','-','black',3)]:
        ax.plot(attribution.date,attribution[column].cumsum(),label=label,linestyle=style,color=color,linewidth=width)
    expiry_code = output_dir.name.removeprefix('expiry_')
    ax.axhline(0,color='#667085',linestyle='--',linewidth=.8); ax.set_title(f'Cumulative PnL Attribution ({expiry_code} Expiry)')
    ax.set_xlabel('Date');ax.set_ylabel('Cumulative PnL');ax.grid(alpha=.3)
    handles,labels=ax.get_legend_handles_labels()
    if capital_daily is not None and not capital_daily.empty:
        capital=capital_daily.copy(); capital['date']=pd.to_datetime(capital.date)
        capital=capital.sort_values('date')
        capital_axis=ax.twinx()
        capital_shadow=capital_axis.fill_between(
            capital.date,0.0,capital.TOTAL_CAPITAL_OCCUPIED.astype(float),
            color='#90a4ae',alpha=.22,label='CAPITAL OCCUPIED',zorder=0,
        )
        capital_axis.set_ylabel('Capital Occupied')
        capital_axis.grid(False)
        handles.append(capital_shadow); labels.append('CAPITAL OCCUPIED')
    ax.legend(handles,labels,ncol=3,fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'));fig.autofmt_xdate();fig.tight_layout()
    fig.savefig(figure_dir/'traditional_taylor_cumulative_attribution.png',dpi=200,bbox_inches='tight');plt.close(fig)

    # Generate one cumulative PnL chart for every Traditional Taylor component and reconciliation item.
    pnl_component_dir=figure_dir/'pnl_components'; pnl_component_dir.mkdir(parents=True,exist_ok=True)
    individual_pnl_columns=[*TRADITIONAL_TAYLOR_COMPONENTS,'MODEL_RESIDUAL','MARKET_NOISE',
                            'TOTAL_RESIDUAL','FEE_PNL','MODEL_PNL','MARKET_PNL',
                            'ACTUAL_PNL_AFTER_FEES']
    for column in individual_pnl_columns:
        figure,axis=plt.subplots(figsize=(11,5.5))
        axis.plot(attribution.date,attribution[column].cumsum(),color='#3568b8',linewidth=2.2)
        axis.axhline(0,color='#667085',linestyle='--',linewidth=.8)
        axis.set_title(f'Cumulative Traditional Taylor: {column}',fontweight='bold')
        axis.set_xlabel('Date');axis.set_ylabel('Cumulative PnL');axis.grid(alpha=.3)
        axis.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
        figure.autofmt_xdate();figure.tight_layout()
        figure.savefig(pnl_component_dir/f'{column.lower()}_cumulative_pnl.png',dpi=200,bbox_inches='tight')
        plt.close(figure)
    return attribution,summary,traditional_exposures

## 9. Run six expiries and create cross-expiry charts


In [19]:
def positions_with_half_skew_vanna_similarity(positions: pd.DataFrame) -> pd.DataFrame:
    """Return positions with daily option Skew/Vanna cross-sectional similarity.

    Unit exposures use one option contract: option multiplier * Greek,
    without qty. Similarities are
    calculated from selected option Greeks and repeated on every date row.
    """
    enriched=positions.copy(); enriched['date']=pd.to_datetime(enriched.date)
    enriched['expiry']=pd.to_datetime(enriched.expiry)
    enriched['OPTION_LEG_UNIT_CALL_SKEW_EXPOSURE']=np.nan
    enriched['OPTION_LEG_UNIT_PUT_SKEW_EXPOSURE']=np.nan
    enriched['OPTION_LEG_UNIT_ATMVOL_VANNA_EXPOSURE']=np.nan
    pars={(x.TRADE_DT,x.EXPIRY):x for x in PAR.itertuples()
          if not str(x.FIT_STATUS).startswith(('FAILED','INSUFFICIENT'))}
    daily=[]
    for date,all_positions in enriched.groupby('date'):
        date=pd.Timestamp(date); option_positions=all_positions[all_positions.asset_type.eq('OPTION')]
        call_pearson=np.nan; call_cosine=np.nan; put_pearson=np.nan; put_cosine=np.nan
        if not option_positions.empty:
            expiry=pd.Timestamp(option_positions.expiry.iloc[0]); model=pars.get((date,expiry))
            quotes=OPT[(OPT.TRADE_DT.eq(date))&OPT.EXPIRY.eq(expiry)].set_index('CODE')
            call_skew_greeks=[]; put_skew_greeks=[]; vanna_greeks=[]
            if model is not None:
                for position_index,leg in option_positions.iterrows():
                    if leg['code'] not in quotes.index: continue
                    row=quotes.loc[leg['code']]
                    if isinstance(row,pd.DataFrame): row=row.iloc[0]
                    risk_vector,_,_=risks(row,model)
                    call_skew_greeks.append(float(risk_vector[3])); put_skew_greeks.append(float(risk_vector[4])); vanna_greeks.append(float(risk_vector[5]))
                    enriched.loc[position_index,'OPTION_LEG_UNIT_CALL_SKEW_EXPOSURE']=M*float(risk_vector[3])
                    enriched.loc[position_index,'OPTION_LEG_UNIT_PUT_SKEW_EXPOSURE']=M*float(risk_vector[4])
                    enriched.loc[position_index,'OPTION_LEG_UNIT_ATMVOL_VANNA_EXPOSURE']=M*float(risk_vector[5])
            call_vector=np.asarray(call_skew_greeks,dtype=float); put_vector=np.asarray(put_skew_greeks,dtype=float); vanna_vector=np.asarray(vanna_greeks,dtype=float)
            if len(call_vector)>=2 and np.std(call_vector)>1e-15 and np.std(vanna_vector)>1e-15:
                call_pearson=float(np.corrcoef(call_vector,vanna_vector)[0,1])
            if len(put_vector)>=2 and np.std(put_vector)>1e-15 and np.std(vanna_vector)>1e-15:
                put_pearson=float(np.corrcoef(put_vector,vanna_vector)[0,1])
            call_denominator=float(np.linalg.norm(call_vector)*np.linalg.norm(vanna_vector))
            put_denominator=float(np.linalg.norm(put_vector)*np.linalg.norm(vanna_vector))
            if call_denominator>1e-15: call_cosine=float(call_vector@vanna_vector/call_denominator)
            if put_denominator>1e-15: put_cosine=float(put_vector@vanna_vector/put_denominator)
        daily.append({'date':date,'CALL_SKEW_VANNA_PEARSON_CORRELATION':call_pearson,
                      'CALL_SKEW_VANNA_COSINE_SIMILARITY':call_cosine,
                      'PUT_SKEW_VANNA_PEARSON_CORRELATION':put_pearson,
                      'PUT_SKEW_VANNA_COSINE_SIMILARITY':put_cosine})
    return enriched.merge(pd.DataFrame(daily),on='date',how='left')


def plot_all_expiry_comparisons(results: Dict[str,Dict[str,pd.DataFrame]], output_dir: Path) -> None:
    output_dir=Path(output_dir); output_dir.mkdir(parents=True,exist_ok=True)
    chart_specs=[
        ('cumulative_actual_pnl_all_expiries.png','Cumulative Actual PnL After Fees','ACTUAL_PNL',True),
        ('cumulative_call_skew_pnl_all_expiries.png','Cumulative Traditional Call Skew PnL','CALL_SKEW_PNL',True),
        ('daily_actual_pnl_all_expiries.png','Daily Actual PnL After Fees','ACTUAL_PNL',False),
        ('daily_call_skew_pnl_all_expiries.png','Daily Traditional Call Skew PnL','CALL_SKEW_PNL',False),
    ]
    for filename,title,column,cumulative in chart_specs:
        fig,ax=plt.subplots(figsize=(15,7.5))
        for code,item in results.items():
            frame=item['pnl'] if column=='ACTUAL_PNL' else item['traditional']; dates=pd.to_datetime(frame.date); values=frame[column].cumsum() if cumulative else frame[column]
            ax.plot(dates,values,label=code,linewidth=2)
        ax.axhline(0,color='#667085',linestyle='--',linewidth=.8); ax.set_title(title,fontweight='bold'); ax.set_xlabel('Date'); ax.set_ylabel('PnL'); ax.grid(alpha=.3); ax.legend(ncol=3,title='Expiry')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d')); fig.autofmt_xdate(); fig.tight_layout(); fig.savefig(output_dir/filename,dpi=200,bbox_inches='tight'); plt.close(fig)

def summarize_capital_return(expiry_code: str, pnl: pd.DataFrame, margin_daily: pd.DataFrame, traditional: pd.DataFrame) -> Dict[str, object]:
    """Summarize capital returns and Traditional-Taylor Half-Skew PnL contributions."""
    capital=margin_daily.copy(); capital['date']=pd.to_datetime(capital.date); capital=capital.sort_values('date')
    opening_row=capital.iloc[0]; maximum_row=capital.loc[capital.TOTAL_CAPITAL_OCCUPIED.astype(float).idxmax()]
    cumulative_actual_pnl=float(pnl.ACTUAL_PNL.sum()); opening_capital=float(opening_row.TOTAL_CAPITAL_OCCUPIED); maximum_capital=float(maximum_row.TOTAL_CAPITAL_OCCUPIED)
    end_date=pd.to_datetime(pnl.date).max().normalize(); holding_days=max((end_date-pd.Timestamp(opening_row.date).normalize()).days,1)
    opening_return=cumulative_actual_pnl/opening_capital if opening_capital>0 else np.nan
    annualized_opening_return=(1.0+opening_return)**(365.0/holding_days)-1.0 if np.isfinite(opening_return) and opening_return>-1.0 else np.nan
    maximum_return=cumulative_actual_pnl/maximum_capital if maximum_capital>0 else np.nan
    annualized_maximum_return=(1.0+maximum_return)**(365.0/holding_days)-1.0 if np.isfinite(maximum_return) and maximum_return>-1.0 else np.nan
    call_skew_pnl=float(traditional['CALL_SKEW_PNL'].sum()); put_skew_pnl=float(traditional['PUT_SKEW_PNL'].sum())
    skew_pnl=call_skew_pnl+put_skew_pnl; cumulative_model_pnl=float(pnl['MODEL_PNL'].sum())
    skew_to_actual_pnl=skew_pnl/cumulative_actual_pnl if abs(cumulative_actual_pnl)>1e-12 else np.nan
    skew_to_model_pnl=skew_pnl/cumulative_model_pnl if abs(cumulative_model_pnl)>1e-12 else np.nan
    skew_absolute,explicit_abs_pnl,ranked_non_skew=traditional_absolute_attribution_breakdown(traditional)
    absolute_skew_share=skew_absolute/explicit_abs_pnl if explicit_abs_pnl>0 else np.nan
    result={'EXPIRY_CODE':str(expiry_code),'BACKTEST_START_DATE':pd.to_datetime(pnl.date).min().date(),
            'BACKTEST_END_DATE':end_date.date(),'HOLDING_CALENDAR_DAYS':holding_days,'CUMULATIVE_ACTUAL_PNL':cumulative_actual_pnl,
            'OPENING_CAPITAL_DATE':opening_row.date.date(),'OPENING_CAPITAL_OCCUPIED':opening_capital,
            'OPENING_CAPITAL_RETURN':opening_return,'ANNUALIZED_OPENING_CAPITAL_RETURN':annualized_opening_return,
            'MAX_CAPITAL_DATE':maximum_row.date.date(),'MAX_CAPITAL_OCCUPIED':maximum_capital,
            'MAX_CAPITAL_RETURN':maximum_return,'ANNUALIZED_MAX_CAPITAL_RETURN':annualized_maximum_return,
            'CUMULATIVE_CALL_SKEW_PNL':call_skew_pnl,'CUMULATIVE_PUT_SKEW_PNL':put_skew_pnl,'CUMULATIVE_SKEW_PNL':skew_pnl,
            'SKEW_TO_ACTUAL_PNL':skew_to_actual_pnl,'SKEW_TO_MODEL_PNL':skew_to_model_pnl,'ABSOLUTE_SKEW_ATTRIBUTION_SHARE':absolute_skew_share}
    for rank,(name,value,share) in enumerate(ranked_non_skew[:3],start=1):
        result[f'TOP_{rank}_NON_SKEW_FACTOR']=name
        result[f'TOP_{rank}_NON_SKEW_ABS_PNL']=value
        result[f'TOP_{rank}_NON_SKEW_SHARE']=share
    return result

if __name__=='__main__':
    output_root=Path(globals().get('SKEW_STRATEGY_OUTPUT_PATH',_output_root/f'06_skew_strategy_{_model_tag}'))
    output_root.mkdir(parents=True,exist_ok=True); results={}; capital_return_records=[]; all_moneyness_stats=[]
    for expiry_code in map(str,STRATEGY_EXPIRY_CODES):
        output_dir=output_root/f'expiry_{expiry_code}'; output_dir.mkdir(parents=True,exist_ok=True)
        pnl,positions,moneyness_stats=run(expiry_code=expiry_code,bound=int(STRATEGY_POSITION_BOUND),minvol=int(STRATEGY_MIN_OPTION_VOLUME),kmax=float(STRATEGY_MAX_ABS_LOG_MONEYNESS),ridge=float(STRATEGY_RIDGE),max_options=MAX_OPTIONS,use_premium_margin_filter=bool(STRATEGY_USE_PREMIUM_MARGIN_FILTER),min_premium_margin_ratio=float(STRATEGY_MIN_PREMIUM_MARGIN_RATIO))
        if pnl.empty: print(f'{expiry_code}: no valid backtest periods'); continue
        pnl.to_csv(output_dir/'daily_account_pnl.csv',index=False); positions.to_csv(output_dir/'positions.csv',index=False)
        moneyness_stats.to_csv(output_dir/'selected_log_moneyness_daily.csv',index=False)
        all_moneyness_stats.append(moneyness_stats)
        positions_similarity=positions_with_half_skew_vanna_similarity(positions)
        positions_similarity.to_csv(output_dir/'positions_with_half_skew_vanna_similarity.csv',index=False)
        option_margin_by_leg,option_margin_daily=calculate_option_margin(positions); futures_margin_by_leg,futures_margin_daily=calculate_futures_margin(positions)
        option_margin_by_leg.to_csv(output_dir/'option_margin_by_leg.csv',index=False); option_margin_daily.to_csv(output_dir/'option_margin_daily.csv',index=False); futures_margin_by_leg.to_csv(output_dir/'futures_margin_by_leg.csv',index=False); futures_margin_daily.to_csv(output_dir/'futures_margin_daily.csv',index=False)
        margin_daily=option_margin_daily.merge(futures_margin_daily,on='date',how='outer').fillna(0.)
        margin_daily['TOTAL_MARGIN']=margin_daily.OPTION_MARGIN+margin_daily.FUTURES_MARGIN
        margin_daily['TOTAL_CAPITAL_OCCUPIED']=margin_daily.LONG_OPTION_PREMIUM+margin_daily.TOTAL_MARGIN
        margin_daily.to_csv(output_dir/'total_margin_daily.csv',index=False)
        plot_cumulative_pnl(pnl,output_dir/'figures'); plot_selected_expiry_half_skew_segments(positions,PAR,output_dir/'figures'/'skew_diagnostics')
        t2,t2_summary,t2_exposures=calculate_t2_attribution(positions,pnl,output_dir,h=float(STRATEGY_TAYLOR_STEP))
        traditional,traditional_summary,traditional_exposures=calculate_traditional_taylor_attribution(positions,pnl,output_dir,capital_daily=margin_daily)
        capital_return_records.append(summarize_capital_return(expiry_code,pnl,margin_daily,traditional))
        results[expiry_code]={'pnl':pnl,'traditional':traditional,'t2':t2}
        print(f'{expiry_code}: {pnl.date.min()} to {pnl.date.max()}, periods={len(pnl)}, actual PnL={pnl.ACTUAL_PNL.sum():,.2f}')
    if all_moneyness_stats:
        pd.concat(all_moneyness_stats,ignore_index=True).to_csv(output_root/'selected_log_moneyness_daily_all_expiries.csv',index=False)

    capital_return_summary=pd.DataFrame(capital_return_records)
    capital_return_summary_for_export=capital_return_summary.rename(columns={
        'EXPIRY_CODE':'到期月份','BACKTEST_START_DATE':'回测起始日','BACKTEST_END_DATE':'回测结束日','HOLDING_CALENDAR_DAYS':'持有自然日数',
        'CUMULATIVE_ACTUAL_PNL':'累计实际盈亏','OPENING_CAPITAL_DATE':'开仓占资日期','OPENING_CAPITAL_OCCUPIED':'开仓占资',
        'OPENING_CAPITAL_RETURN':'开仓占资收益率','ANNUALIZED_OPENING_CAPITAL_RETURN':'开仓占资年化收益率',
        'MAX_CAPITAL_DATE':'最大占资日期','MAX_CAPITAL_OCCUPIED':'最大占资','MAX_CAPITAL_RETURN':'最大占资收益率','ANNUALIZED_MAX_CAPITAL_RETURN':'最大占资年化收益率',
        'CUMULATIVE_CALL_SKEW_PNL':'累计Call Skew PnL','CUMULATIVE_PUT_SKEW_PNL':'累计Put Skew PnL','CUMULATIVE_SKEW_PNL':'累计Skew PnL',
        'SKEW_TO_ACTUAL_PNL':'Skew PnL/实际净PnL','SKEW_TO_MODEL_PNL':'Skew PnL/模型PnL','ABSOLUTE_SKEW_ATTRIBUTION_SHARE':'Skew绝对归因占比',
        'TOP_1_NON_SKEW_FACTOR':'第一大非Skew因素','TOP_1_NON_SKEW_ABS_PNL':'第一大非Skew因素绝对PnL','TOP_1_NON_SKEW_SHARE':'第一大非Skew因素占比',
        'TOP_2_NON_SKEW_FACTOR':'第二大非Skew因素','TOP_2_NON_SKEW_ABS_PNL':'第二大非Skew因素绝对PnL','TOP_2_NON_SKEW_SHARE':'第二大非Skew因素占比',
        'TOP_3_NON_SKEW_FACTOR':'第三大非Skew因素','TOP_3_NON_SKEW_ABS_PNL':'第三大非Skew因素绝对PnL','TOP_3_NON_SKEW_SHARE':'第三大非Skew因素占比',
    })
    _return_columns=['开仓占资收益率','开仓占资年化收益率','最大占资收益率','最大占资年化收益率','Skew PnL/实际净PnL','Skew PnL/模型PnL','Skew绝对归因占比','第一大非Skew因素占比','第二大非Skew因素占比','第三大非Skew因素占比']
    _amount_columns=['累计实际盈亏','开仓占资','最大占资','累计Call Skew PnL','累计Put Skew PnL','累计Skew PnL','第一大非Skew因素绝对PnL','第二大非Skew因素绝对PnL','第三大非Skew因素绝对PnL']
    capital_return_summary_for_export[_return_columns]=capital_return_summary_for_export[_return_columns].map(lambda x:f'{x:.2%}' if pd.notna(x) else '')
    capital_return_summary_for_export[_amount_columns]=capital_return_summary_for_export[_amount_columns].round(2)
    capital_return_summary_for_export.to_csv(output_root/'capital_return_summary.csv',index=False,encoding='utf-8-sig',float_format='%.2f')
    print('\nCapital return summary (returns shown as percentages):')
    print(capital_return_summary.to_string(index=False,formatters={
        'CUMULATIVE_ACTUAL_PNL':lambda x:f'{x:,.2f}',
        'OPENING_CAPITAL_OCCUPIED':lambda x:f'{x:,.2f}',
        'OPENING_CAPITAL_RETURN':lambda x:f'{x:.2%}',
        'ANNUALIZED_OPENING_CAPITAL_RETURN':lambda x:f'{x:.2%}',
        'MAX_CAPITAL_OCCUPIED':lambda x:f'{x:,.2f}',
        'MAX_CAPITAL_RETURN':lambda x:f'{x:.2%}',
        'ANNUALIZED_MAX_CAPITAL_RETURN':lambda x:f'{x:.2%}',
        'CUMULATIVE_CALL_SKEW_PNL':lambda x:f'{x:,.2f}',
        'CUMULATIVE_PUT_SKEW_PNL':lambda x:f'{x:,.2f}',
        'CUMULATIVE_SKEW_PNL':lambda x:f'{x:,.2f}',
        'SKEW_TO_ACTUAL_PNL':lambda x:f'{x:.2%}',
        'SKEW_TO_MODEL_PNL':lambda x:f'{x:.2%}',
        'ABSOLUTE_SKEW_ATTRIBUTION_SHARE':lambda x:f'{x:.2%}',
        'TOP_1_NON_SKEW_ABS_PNL':lambda x:f'{x:,.2f}','TOP_1_NON_SKEW_SHARE':lambda x:f'{x:.2%}',
        'TOP_2_NON_SKEW_ABS_PNL':lambda x:f'{x:,.2f}','TOP_2_NON_SKEW_SHARE':lambda x:f'{x:.2%}',
        'TOP_3_NON_SKEW_ABS_PNL':lambda x:f'{x:,.2f}','TOP_3_NON_SKEW_SHARE':lambda x:f'{x:.2%}',
    }))
    plot_all_expiry_comparisons(results,output_root/'figures'/'all_expiries')

2606: no valid backtest periods


2607: 2026-06-02 00:00:00 to 2026-06-26 00:00:00, periods=18, actual PnL=187,776.03


2608: 2026-06-24 00:00:00 to 2026-07-29 00:00:00, periods=26, actual PnL=116,098.34


2609: 2026-06-02 00:00:00 to 2026-07-29 00:00:00, periods=41, actual PnL=126,414.50


2612: 2026-06-02 00:00:00 to 2026-07-29 00:00:00, periods=41, actual PnL=265,714.62


2703: 2026-07-21 00:00:00 to 2026-07-29 00:00:00, periods=7, actual PnL=145,367.54

Capital return summary (returns shown as percentages):
EXPIRY_CODE BACKTEST_START_DATE BACKTEST_END_DATE  HOLDING_CALENDAR_DAYS CUMULATIVE_ACTUAL_PNL OPENING_CAPITAL_DATE OPENING_CAPITAL_OCCUPIED OPENING_CAPITAL_RETURN ANNUALIZED_OPENING_CAPITAL_RETURN MAX_CAPITAL_DATE MAX_CAPITAL_OCCUPIED MAX_CAPITAL_RETURN ANNUALIZED_MAX_CAPITAL_RETURN CUMULATIVE_CALL_SKEW_PNL CUMULATIVE_PUT_SKEW_PNL CUMULATIVE_SKEW_PNL SKEW_TO_ACTUAL_PNL SKEW_TO_MODEL_PNL ABSOLUTE_SKEW_ATTRIBUTION_SHARE TOP_1_NON_SKEW_FACTOR TOP_1_NON_SKEW_ABS_PNL TOP_1_NON_SKEW_SHARE TOP_2_NON_SKEW_FACTOR TOP_2_NON_SKEW_ABS_PNL TOP_2_NON_SKEW_SHARE TOP_3_NON_SKEW_FACTOR TOP_3_NON_SKEW_ABS_PNL TOP_3_NON_SKEW_SHARE
       2607          2026-06-02        2026-06-26                     25            187,776.03           2026-06-01             3,420,100.35                  5.49%                           118.23%       2026-06-24        21,176,553.82     